# 02b · Add more items to the sample you already have

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/02b_add_samples.ipynb)

More annotated items is more evidence — drawn without repeating what you have.

```
  01_build_pool_<track>  →  02_sample  →▶ 02b_add_samples  →  03_annotate  →  04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/pools/<track>_pool.json`, `data/gold/<track>_<group>_sample.json`, and the sheet link from 02 |
| **Writes** | the same `_sample.json`, enlarged, and new rows in the sheet you are already annotating in |

---

**Open this only if you have time left over**, and only once the first draw is annotated or nearly so. It does not replace `02_sample.ipynb` and it does not make a second sheet — it adds rows to the bottom of the one your group is already working in.

Nothing already in the sheet is renumbered, moved or overwritten. The new items get ids that carry on from the highest one you have.

**One person runs this.** Everyone else can keep annotating in the sheet while it runs; the new rows appear below the ones they are working on.

> If you moved columns around in the sheet, or added your own, that is fine — every column is found by the name in row 1. What is not fine is renaming `ID`, `Text`, `Label` or `Note`: notebook 03 needs those names too.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH,
                    DEV_PATH, TEST_PATH, DISAGREED_PATH, PRED_PATH,
                    ROUNDS_PATH, TESTLOG_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

# Same split as notebook 02: reading and writing files is plumbing, so it is
# imported, and the sampling you have to defend is defined further down.
from pipeline import load_gold, save_json, label_set
from annotate import (append_to_annotation_sheet, remembered_sheet,
                      tab_names, load_coder_sheets)

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## What a second draw does, and what it does not

More annotated items is more evidence: your F1 rests on a bigger sample and your agreement number gets steadier. That is real, and it is why this notebook exists.

Be careful about one thing when you write it up. Your gold set was now built in **two draws**, and they are only one sample of the pool if you drew them the same way, for the same reason. So:

- **Same strategy, more items.** Say how many you drew in each round and why you stopped where you did. This is the simple case.
- **A different strategy the second time.** That is a two-stage design, and your report has to describe it as one. It is a legitimate choice — starting balanced and topping up by document is a real thing to want — but it is not something to leave for the reader to notice from the counts.

Either way, both rounds go in `PLAN.md`: strategy, size, seed, and the reason.

> **A label that has run out.** If the pool has no items left under some label, a balanced top-up simply will not contain it and your combined sample stops being balanced. The cells below say so when it happens. That belongs in your limitations — do not switch strategy to hide it.

First we open the pool and the sample you already have. Both come off disk: the session that drew the first sample is long gone, and this is what it left behind.

In [ ]:
pool = load_gold(POOL_PATH)
sampled = load_gold(SAMPLE_PATH)

highest = 0
for item in sampled:
    highest = max(highest, int(item["id"]))

print("pool:", len(pool), "items")
print("already sampled:", len(sampled), "items, ids 1 to", highest)

Now we find the sheet your group has been annotating in, and list its tabs. Read that list before going on: the new rows are added to exactly these tabs, and a coder whose tab is not named in `CODERS` in `config.yaml` would not get them.

In [ ]:
SHEET_ID = remembered_sheet(SHEET_PATH)
print("sheet:", SHEET_ID)
print("tabs in it:", tab_names(SHEET_ID))
print("tabs the new rows will go in:", list(CODERS), "+ Final")

## The code that draws the extra items — read it, then run it

Two functions. Neither draws anything new by itself: the actual drawing is done by the same `sample` you read and ran in notebook 02, over a smaller pool.

Run `help(sample)` if you want the three strategies again — they have not changed. To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left.

First, the helpers the definitions below call. Nothing to decide here — run it and read on.

In [ ]:
from pipeline import _refuse_overlapping_draw, label_set, reid, sample

**`remaining_pool`** is the answer to *which items have we already got?* — which is harder than it sounds, because the ids in your sample file are 1, 2, 3 … and the ids in the pool are not. It matches on `source_id`, the id each item had in the pool, and on the text as well. An item that matches either is left out, so nothing you have annotated can be drawn twice.

In [ ]:
def remaining_pool(pool: list[dict[str, str]],
                   sampled: list[dict[str, str]]) -> list[dict[str, str]]:
    """The pool items you have NOT already drawn.

    Matched two ways, and an item is excluded if EITHER matches:

      source_id  the id the item had in the pool, recorded when it was drawn.
      text       for samples drawn before source_id was recorded, and as a check on
                 a pool that has been rebuilt with different ids since you drew.

    Args:
        pool: the full pool for your track.
        sampled: the items you already have, read back from your sample file.

    Returns:
        The pool items that are in neither of those, in pool order.

    Example:
        >>> left = remaining_pool(pool, sampled)
    """
    taken_ids = set()
    taken_texts = set()
    without_source_id = 0
    for item in sampled:
        if "source_id" in item:
            taken_ids.add(item["source_id"])
        else:
            without_source_id = without_source_id + 1
        taken_texts.add(str(item["text"]))

    remaining = []
    for item in pool:
        if item.get("id") in taken_ids:
            continue
        if str(item["text"]) in taken_texts:
            continue
        remaining.append(item)

    excluded = len(pool) - len(remaining)
    print("pool:", len(pool), "· already drawn:", len(sampled),
          "· left to draw from:", len(remaining))

    if without_source_id > 0:
        print("NOTE:", without_source_id, "of your", len(sampled), "sampled items carry")
        print("      no source_id, so they were matched to the pool by their text.")

    # More pool rows excluded than you drew means the pool holds the same text twice.
    # Nothing you have annotated can be drawn again, which is the direction you want,
    # but it does take items out of reach - so say it rather than let the count puzzle
    # someone later.
    if excluded > len(sampled):
        print("NOTE:", excluded, "pool items were excluded but you have only",
              len(sampled), "sampled.")
        print("      The pool holds some texts more than once, so the copies of what")
        print("      you already drew were excluded too.")
    return remaining

**`sample_more`** is the one you call. In: the pool, the items you already have, and the same three arguments as notebook 02. Out: **only the new items**, numbered on from your highest id. Step 3 is one call to the same `sample` you already read; the rest is checking. Step 5 refuses outright if any new item repeats an id, a `source_id` or a text you already have — two rows sharing an id are merged into one when the sheet is read back, which would cost you annotation you had already done.

In [ ]:
def sample_more(pool: list[dict[str, str]],
                sampled: list[dict[str, str]],
                strategy: str,
                n_per_class: int,
                seed: int = 42,
                n_per_doc: int = 4) -> list[dict[str, str]]:
    """Draw MORE items, from the part of the pool you have not drawn yet.

    The same three strategies as your first draw, sized the same way from n_per_class.
    The choice is yours again and it needs the same one-sentence reason in PLAN.md -
    and if you pick a different strategy this time, your gold set was built in two
    stages and the report has to say so.

    Args:
        pool: the full pool for your track.
        sampled: the items you already have, read back from your sample file.
        strategy: "balanced", "random" or "by_document", as in notebook 02.
        n_per_class: how many more items per label.
        seed: use a different one from your first draw, and record both.
        n_per_doc: how many sentences per document, for "by_document".

    Returns:
        Only the NEW items. Their ids carry on from the highest id you already have,
        so nothing in your sheet is renumbered and no two rows share an id.

    Raises:
        ValueError: when `sampled` is empty, when the pool is used up, or when the
            new items collide with the ones you already have.

    Example:
        >>> extra = sample_more(pool, sampled, "balanced", 5, seed=SEED + 1)
    """
    if not sampled:
        raise ValueError(
            "sample_more adds to a draw you already have, and the list you passed is "
            "empty.\n"
            "If this is your first draw, use 02_sample.ipynb instead. If it is not, "
            "the sample file did not load - check the cell above.")

    ### Step 1: what is left ###
    remaining = remaining_pool(pool, sampled)
    if not remaining:
        raise ValueError(
            "Every item in the pool is already in your sample, so there is nothing "
            "left to add.\n"
            "This is the end of what this track can give you. Say so in the report - "
            "a sample that is the whole pool is a census, not a sample.")

    ### Step 2: has a label run out? ###
    # Worth saying BEFORE the draw. A balanced top-up simply will not contain that
    # label, and a random one is sized from the labels that are left, so it comes out
    # smaller than the same call on the full pool would have given.
    gone = []
    labels_left = label_set(remaining)
    for label in label_set(pool):
        if label not in labels_left:
            gone.append(label)
    if gone:
        print("NOTE: no items left in the pool for:", ", ".join(gone))
        print("      A balanced top-up will not contain them, and a random one is")
        print("      sized from the", len(labels_left), "labels that are left.")
        print("      Your combined sample stops being balanced. That belongs in the")
        print("      limitations of your report, not in a change of strategy.")

    ### Step 3: the same draw you made the first time, over what is left ###
    extra = sample(remaining, strategy, n_per_class, seed, n_per_doc)

    ### Step 4: ids that carry on from the ones you already have ###
    # Not from 1. Two rows sharing an id are merged into one when the sheet is read
    # back, so a restart here would quietly cost you half your annotation.
    highest = 0
    for item in sampled:
        highest = max(highest, int(item["id"]))
    extra = reid(extra, start=highest + 1)

    ### Step 5: check the new items really are new ###
    _refuse_overlapping_draw(extra, sampled)

    last = extra[len(extra) - 1]["id"]
    print("")
    print("Adding", len(extra), "items, ids", str(extra[0]["id"]) + ".." + str(last) +
          ". The", len(sampled), "you already have keep theirs.")
    return extra

Run this to check the definitions above took effect. It prints the first line of the function the notebook will actually use, and the description of each argument.

In [ ]:
help(sample_more)

### Choose a strategy again — the same decision as notebook 02

| strategy | what it draws | what it means the second time |
|---|---|---|
| `balanced` | up to `n` more of **each** label | keeps the combined sample level, as long as no label has run out |
| `random` | the same total, ignoring labels | keeps the pool's own imbalance; the top-up looks like the corpus |
| `by_document` | whole passages (`cars50` · `raamove`) | new documents, so the combined sample covers more texts |

Use a **different seed** from your first draw. Both go in the report: a sample nobody can redraw is a sample nobody can check, and there are two draws to redraw now.

Write the second strategy and the reason in `PLAN.md` §5, beside the first — **even if it is the same word.** "We drew 20 more the same way, because we had time and wanted a steadier κ" is a complete answer; leaving the section as it was is not, because it now describes half of what you did.

This cell only draws. Nothing reaches the sheet or the disk until the steps below, so run it, read the counts, and run it again with different numbers if you do not like them.

In [ ]:
# ══ STEP 1 · Draw the extra items ═════════════════════════════════════════
# Draws more items from the part of the pool you have not already sampled, and
# numbers them on from your highest id. Nothing is written yet.
# Creates: extra

# ✏️ this runs as written — the work is deciding whether it should

# The same one-word choice as notebook 02, made again.
STRATEGY = "balanced"     # "balanced" · "random" · "by_document"
N_MORE = 5                # how many MORE per label

# A different seed from your first draw. Record both in the report.
extra = sample_more(pool, sampled, STRATEGY, N_MORE, SEED + 1)


### Check the draw before it touches anything

The same three questions as notebook 02, asked of the combined sample this time. If the counts are not what you wanted, change the numbers above and run that cell again — nothing has been written yet.

In [ ]:
# ══ STEP 2 · Check what you drew ══════════════════════════════════════════
# Prints the per-label counts of the new items, the ones you already had, and
# the two together. Nothing new is named — this is a check.

# ✏️ this runs as written — the work is deciding whether it should

def count_labels(items):
    counts = {}
    for item in items:
        label = item["label"]
        if label not in counts:
            counts[label] = 0
        counts[label] = counts[label] + 1
    return counts

print("you had: ", count_labels(sampled))
print("adding:  ", count_labels(extra))
print("total:   ", count_labels(sampled + extra))
print("left over for few-shot examples:",
      len(pool) - len(sampled) - len(extra))


## Now write it down — sheet first, file second

The next three cells do the writing, and the order matters.

1. **Keep a copy** of the sample as it is now, so a top-up that goes wrong can be undone.
2. **Add the rows to the sheet.** Every tab is checked first, and if any tab cannot be written to safely then *no* tab is written to — you get the rows printed out to paste in by hand instead.
3. **Save the enlarged sample file.**

The sheet goes before the file on purpose. If the sheet write fails, nothing on disk has changed and you can simply run these cells again. If it succeeds and the file save then fails, the two disagree — but notebook 03 *tells you* so, because it matches the sheet's rows against this file by id. The other order would leave you with a file claiming rows the sheet never got, which shows up as rows that stay blank forever and nothing saying why.

In [ ]:
# ══ STEP 3 · Keep a copy of the sample as it is ═══════════════════════════
# Writes the sample as it stands now to a second file, before step 5 replaces
# the original. This also refuses if you have run this notebook before.

# ✏️ this runs as written — the work is deciding whether it should

save_json(sampled, SAMPLE_BEFORE_TOPUP_PATH,
          what="the sample before the top-up")


If that cell refused, you have run this notebook before. Do not pass `overwrite=True` to get past it without checking — read the sheet first and work out whether the rows are already in there. Adding them twice is much harder to undo than working out what happened.

In [ ]:
# ══ STEP 4 · Add the rows to the sheet ════════════════════════════════════
# Adds one row per new item to the bottom of every coder tab and the Final tab.
# Checks every tab first: if any of them fails, nothing is written anywhere.

# ✏️ this runs as written — the work is deciding whether it should

append_to_annotation_sheet(SHEET_ID, extra,
                           coders=CODERS,
                           # catches a sheet that has drifted from the file
                           expected_rows=len(sampled))


The rows are in the sheet, so now the file has to match them. This is the one cell in the project that overwrites a file you already have, and it is why step 3 kept a copy first.

In [ ]:
# ══ STEP 5 · Save the enlarged sample ═════════════════════════════════════
# Replaces the sample file with the old items plus the new ones.
# Overwriting is deliberate here — step 3 kept the copy.
# Creates: sampled_all

# ✏️ this runs as written — the work is deciding whether it should

sampled_all = sampled + extra
save_json(sampled_all, SAMPLE_PATH, what="sampled items",
          overwrite=True)   # step 3 kept the old one


### Check the sheet and the file agree

This reads both back and compares them. It is the one check that the file notebook 03 will work from holds the same items as the sheet you are annotating in.

A ✗ on the last line is the serious one: two rows sharing an id are merged into one when the sheet is read back, so it would quietly cost you annotation.

In [ ]:
# ══ STEP 6 · Read both back and compare ═══════════════════════════════════
# Reads the sheet and the sample file off disk and checks they hold the same
# ids. Prints a ✓ or a ✗ per check. Nothing new is named — this is a check.

# ✏️ this runs as written — the work is deciding whether it should

rows = load_coder_sheets(SHEET_ID, CODERS)
on_disk = load_gold(SAMPLE_PATH)

sheet_ids = []
for row in rows:
    sheet_ids.append(int(row["ID"]))
file_ids = []
for item in on_disk:
    file_ids.append(int(item["id"]))

def tick(passed):
    return "✓" if passed else "✗"

print(tick(len(sheet_ids) == len(file_ids)),
      "sheet has", len(sheet_ids), "rows · file has", len(file_ids))
print(tick(set(sheet_ids) == set(file_ids)),
      "the sheet and the file hold the same ids")
print(tick(len(sheet_ids) == len(set(sheet_ids))),
      "no id appears twice in the sheet")


---

## 🛑 This notebook is finished. Go and annotate the new rows.

- **Annotate the new rows only.** Everything above them is already done.
- **Spell the labels the same way** as the first round. `to_canonical` will tell you about a typo, but a label spelled two ways is two labels until then.
- **Check the drop-down reached the new rows.** If your `Label` column has one, click a `Label` cell in a new row. Google Sheets does not always carry a drop-down down to rows added later; if it is missing, copy a `Label` cell from a row above and paste it over the new ones. Same for any colour rules you set up.
- **Both coders again.** A row labelled by one person does not contribute to agreement, and a gold set where half the items were double-coded and half were not is one you have to explain.

> **Do not run this notebook again**, and do not run `02_sample.ipynb` again either. When the new rows are annotated, go back to `03_annotate.ipynb` — it needs no changes at all, and picks up the enlarged sample and the same sheet.

**Next:** `03_annotate.ipynb`, once the new rows are labelled too.